# 01 — Download one slice, decrypt, discover schema (Phase 1)

**NETWORK REQUIRED — login/JupyterHub node only.** Downloads the first
benchmark slice (`taubench_airline`, 47 files / 3.41 GB, under the 5 GB
guardrail) to `$HAL_DATA_DIR/encrypted/<slice>/`, decrypts in place
(`hal-decrypt` writes json next to the zips), walks the decrypted traces,
and emits `schema/SCHEMA.md` for local review. Downloads are resumable.

In [ ]:
import polars as pl
from halcausal.io import hf

rows = hf.inventory()
pl.DataFrame([{k: r[k] for k in ("benchmark", "presumed", "n_files", "total_gb", "median_mb")} for r in rows])

In [ ]:
SLICE = "taubench_airline"
zips = hf.download_slice(SLICE)  # hard-errors above 5 GB without explicit approval
print(len(zips), "zips,", round(sum(p.stat().st_size for p in zips) / 1e9, 2), "GB")

In [ ]:
from halcausal.io import decrypt

decrypted = decrypt.decrypt_dir(zips[0].parent)
print(len(decrypted), "decrypted json files from", len(zips), "zips")
assert decrypted, "decryption produced nothing — stop and investigate"
if len(decrypted) < len(zips):
    print("WARNING: fewer json than zips — record in reports/discrepancies.md")

In [ ]:
from halcausal.etl import discover_schema

md = discover_schema(zips[0].parent)  # N from configs/env.yaml (schema_sample_traces)
print(md)
print(md.read_text()[:3000])

## Ship the schema for local review

`schema/` is HPC-writable (results-contract exception). ETL stays blocked
until this is reviewed locally and curated into `schema/field_mapping.yaml`.

In [ ]:
!git pull --rebase
!git add schema/
!python scripts/check_results_size.py --staged

In [ ]:
# Run ONLY after the guard prints OK:
# !git commit -m "Phase 1: empirical SCHEMA.md from taubench_airline slice" && git push